# Part 4 | SQL

Run the setup cell first, then run each query with `%%sql`. The SQL targets **Amazon Redshift** and uses the case schema `lucky_case`. The setup registers a notebook-local cell magic and loads the CSVs into an in-memory SQLite test database. The local runner adapts Redshift's date functions, decimal casts and schema prefix. The SQL itself can be copied from either cell below its magic line into Redshift. The saved results were executed locally.

In [1]:
from pathlib import Path
import re
import sqlite3

import pandas as pd
from IPython import get_ipython
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "data").is_dir())
connection = sqlite3.connect(":memory:")

for table in ("users", "transactions", "loans"):
    source, = (ROOT / "data").glob(f"*{table}.csv")
    pd.read_csv(source).to_sql(table, connection, index=False)


def sql_timestamp(value):
    return value.strftime("%Y-%m-%d %H:%M:%S")


def dateadd(unit, count, value):
    if value is None:
        return None
    offset = pd.Timedelta(count, unit={"day": "d", "hour": "h"}[unit])
    return sql_timestamp(pd.Timestamp(value) + offset)


def date_trunc(unit, value):
    if value is None:
        return None
    value = pd.Timestamp(value)
    if unit == "week":
        value = value.normalize() - pd.Timedelta(days=value.weekday())
    elif unit == "month":
        value = value.normalize().replace(day=1)
    elif unit == "day":
        value = value.normalize()
    else:
        raise ValueError(unit)
    return sql_timestamp(value)


connection.create_function("DATEADD", 3, dateadd, deterministic=True)
connection.create_function("DATE_TRUNC", 2, date_trunc, deterministic=True)


def run_sql(line, cell):
    if line.strip():
        raise ValueError("This notebook's %%sql magic takes no arguments.")
    local_sql = re.sub(r"\bDATEADD\(\s*(day|hour)\s*,", r"DATEADD('\1',", cell, flags=re.I)
    local_sql = re.sub(r"\bTIMESTAMP\s+'([^']+)'", r"'\1'", local_sql, flags=re.I)
    local_sql = re.sub(r"\bAS\s+DECIMAL\(\s*\d+\s*,\s*\d+\s*\)", "AS REAL", local_sql, flags=re.I)
    local_sql = re.sub(r"\blucky_case\.", "", local_sql)
    result = pd.read_sql_query(local_sql, connection)
    with pd.option_context("display.max_rows", None):
        display(result)
    return None


get_ipython().register_magic_function(run_sql, magic_kind="cell", magic_name="sql")

## 1. Mature DPD30 by disbursal month and acquisition channel

The Q3 cohort uses the supplied disbursal dates. A loan enters the denominator when its due date plus 30 days is on or before the November 5 extract. Full repayment means principal plus the recorded fee, with the final repayment on or before the cutoff date. The join uses known non-QA borrowers, and blank UTM values remain a distinct attribution bucket. Channel-month groups with zero mature loans have a null rate.

In [2]:
%%sql
WITH clean_users AS (
    SELECT user_id,
           CASE
               WHEN NULLIF(TRIM(utm_source), '') IS NULL THEN 'Unattributed'
               WHEN LOWER(TRIM(utm_source)) IN
                    ('meta', 'facebook ads', 'facebook', 'fb', 'instagram', 'ig') THEN 'Meta'
               WHEN LOWER(TRIM(utm_source)) IN
                    ('google', 'google ads', 'adwords', 'gads') THEN 'Google Ads'
               WHEN LOWER(TRIM(utm_source)) IN ('tiktok', 'tt') THEN 'TikTok'
               WHEN LOWER(TRIM(utm_source)) IN ('referral', 'ref', 'invite') THEN 'Referral'
               WHEN LOWER(TRIM(utm_source)) = 'organic' THEN 'Organic'
               WHEN LOWER(TRIM(utm_source)) = '(direct)' THEN 'Direct'
               ELSE 'Unmapped'
           END AS channel
    FROM lucky_case.users
    WHERE COALESCE(LOWER(TRIM(full_name)), '') NOT IN
          ('qa lucky', 'test user', 'test qa', 'test test', 'تجربه تطبيق', 'تجربة')
),
loan_cohort AS (
    SELECT l.loan_id,
           DATE_TRUNC('month', l.disbursed_date) AS disbursal_month,
           u.channel,
           DATEADD(day, 30, l.due_date) AS dpd30_date,
           DATE_TRUNC('day', l.last_repayment_date) AS last_repayment_day,
           l.amount_repaid_egp,
           l.principal_egp + l.fee_egp AS full_amount_egp
    FROM lucky_case.loans AS l
    JOIN clean_users AS u ON u.user_id = l.user_id
    WHERE DATE_TRUNC('day', l.disbursed_date) >= TIMESTAMP '2026-07-01 00:00:00'
      AND DATE_TRUNC('day', l.disbursed_date) <  TIMESTAMP '2026-10-01 00:00:00'
),
classified AS (
    SELECT *,
           CASE WHEN dpd30_date <= TIMESTAMP '2026-11-05 00:00:00' THEN 1 ELSE 0 END AS mature,
           CASE WHEN amount_repaid_egp >= full_amount_egp - 0.01
                     AND last_repayment_day <= dpd30_date THEN 1 ELSE 0 END AS full_by_dpd30
    FROM loan_cohort
)
SELECT disbursal_month,
       channel,
       COUNT(*) AS disbursed_loans,
       SUM(mature) AS mature_loans,
       SUM(CASE WHEN mature = 1 AND full_by_dpd30 = 0 THEN 1 ELSE 0 END) AS dpd30_defaults,
       ROUND(
           100.0 * CAST(SUM(CASE WHEN mature = 1 AND full_by_dpd30 = 0 THEN 1 ELSE 0 END)
                        AS DECIMAL(18, 4)) / NULLIF(SUM(mature), 0),
           2
       ) AS dpd30_default_pct
FROM classified
GROUP BY disbursal_month, channel
ORDER BY disbursal_month, channel;

,disbursal_month,channel,disbursed_loans,mature_loans,dpd30_defaults,dpd30_default_pct
0,2026-07-01 00:00:00,Direct,20,20,2,10.00
1,2026-07-01 00:00:00,Google Ads,81,81,2,2.47
2,2026-07-01 00:00:00,Meta,52,52,5,9.62
3,2026-07-01 00:00:00,Organic,17,17,2,11.76
4,2026-07-01 00:00:00,Referral,43,43,2,4.65
5,2026-07-01 00:00:00,TikTok,16,16,3,18.75
6,2026-07-01 00:00:00,Unattributed,31,31,1,3.23
7,2026-08-01 00:00:00,Direct,19,19,2,10.53
8,2026-08-01 00:00:00,Google Ads,142,142,5,3.52
9,2026-08-01 00:00:00,Meta,107,107,16,14.95


## 2. Weekly retention by signup cohort

Unit of analysis: a distinct activated user. Match the Q3 board snapshot by using the case README's fixed UTC+2 offset for Q3 membership and Monday signup-week labels. Measure elapsed time from each user's UTC signup. Days 0–6 are week 0, so week 1 starts at day 7. Use half-open windows: [7, 14), [14, 21), [28, 35), and [56, 63) days. Each week's denominator includes activated users whose entire window is observable by the end of the November 5 extract day. The first and last calendar cohorts may contain only the Q3 portion of a week.

Activation uses KYC verified by the extract and at least one successful deposit from signup through day 7, inclusive. Retention counts successful transactions of any type in the target window. Exact transaction copies are removed and user joins exclude QA and unmatched transactions. A day-7 activation deposit can count as week-1 activity. The result shows eligible and retained counts so partial observation can be seen directly.

In [3]:
%%sql
WITH settings AS (
    SELECT TIMESTAMP '2026-11-06 00:00:00' AS extract_end_utc
),
clean_users AS (
    SELECT user_id, signup_ts, kyc_verified_ts
    FROM lucky_case.users
    WHERE COALESCE(LOWER(TRIM(full_name)), '') NOT IN
          ('qa lucky', 'test user', 'test qa', 'test test', 'تجربه تطبيق', 'تجربة')
      -- Cairo Q3 boundaries expressed in UTC using the README's fixed UTC+2.
      AND signup_ts >= TIMESTAMP '2026-06-30 22:00:00'
      AND signup_ts <  TIMESTAMP '2026-09-30 22:00:00'
),
successful_transactions AS (
    SELECT DISTINCT t.txn_id, t.user_id, t.txn_ts, t.txn_type, t.amount_egp, t.status
    FROM lucky_case.transactions AS t
    JOIN clean_users AS u ON u.user_id = t.user_id
    CROSS JOIN settings AS s
    WHERE t.status = 'success'
      AND t.txn_ts < s.extract_end_utc
),
activated AS (
    SELECT u.user_id,
           u.signup_ts,
           DATE_TRUNC('week', DATEADD(hour, 2, u.signup_ts)) AS signup_week
    FROM clean_users AS u
    CROSS JOIN settings AS s
    WHERE u.kyc_verified_ts < s.extract_end_utc
      AND EXISTS (
          SELECT 1
          FROM successful_transactions AS t
          WHERE t.user_id = u.user_id
            AND t.txn_type = 'deposit'
            AND t.txn_ts >= u.signup_ts
            AND t.txn_ts <= DATEADD(day, 7, u.signup_ts)
      )
),
target_weeks AS (
    SELECT 1 AS week_number, 7 AS start_day, 14 AS end_day
    UNION ALL SELECT 2, 14, 21
    UNION ALL SELECT 4, 28, 35
    UNION ALL SELECT 8, 56, 63
),
user_windows AS (
    SELECT a.user_id,
           a.signup_week,
           w.week_number,
           DATEADD(day, w.start_day, a.signup_ts) AS week_start,
           DATEADD(day, w.end_day, a.signup_ts) AS week_end,
           CASE WHEN DATEADD(day, w.end_day, a.signup_ts) <= s.extract_end_utc
                THEN 1 ELSE 0 END AS observable
    FROM activated AS a
    CROSS JOIN target_weeks AS w
    CROSS JOIN settings AS s
),
user_retention AS (
    SELECT w.user_id,
           w.signup_week,
           w.week_number,
           w.observable,
           MAX(CASE WHEN t.user_id IS NOT NULL THEN 1 ELSE 0 END) AS retained
    FROM user_windows AS w
    LEFT JOIN successful_transactions AS t
      ON t.user_id = w.user_id
     AND w.observable = 1
     AND t.txn_ts >= w.week_start
     AND t.txn_ts < w.week_end
    GROUP BY w.user_id, w.signup_week, w.week_number, w.observable
)
SELECT signup_week,
       week_number,
       COUNT(*) AS activated_users,
       SUM(observable) AS observable_users,
       SUM(retained) AS retained_users,
       ROUND(100.0 * CAST(SUM(retained) AS DECIMAL(18, 4))
             / NULLIF(SUM(observable), 0), 2) AS retention_pct
FROM user_retention
GROUP BY signup_week, week_number
ORDER BY signup_week, week_number;

,signup_week,week_number,activated_users,observable_users,retained_users,retention_pct
0,2026-06-29 00:00:00,1,121,121,90,74.38
1,2026-06-29 00:00:00,2,121,121,77,63.64
2,2026-06-29 00:00:00,4,121,121,67,55.37
3,2026-06-29 00:00:00,8,121,121,53,43.80
4,2026-07-06 00:00:00,1,174,174,123,70.69
5,2026-07-06 00:00:00,2,174,174,117,67.24
6,2026-07-06 00:00:00,4,174,174,92,52.87
7,2026-07-06 00:00:00,8,174,174,86,49.43
8,2026-07-13 00:00:00,1,173,173,114,65.90
9,2026-07-13 00:00:00,2,173,173,109,63.01
